# Slide figures

Lecture figures **about our corpora** that the code-along notebooks do not already produce. Run all cells, or `jupyter nbconvert --execute slide-figures.ipynb`, to regenerate them into `figures/`. Instructor tooling, not student-facing.

Right now: the two D2-PM summary-metric plots and the whole-corpus nuclear network — the gaps listed in `docs/2026-slide-translation.md`. The rest of the slide figures come from the code-along notebooks via `tools/export_figures.py`. Add a cell here whenever a slide needs a corpus figure the notebooks don't make.

In [ ]:
from pathlib import Path

import liwc
import matplotlib.pyplot as plt
import sotu

from corpus_tools import draw_cooccurrence_network, liwcalike, load_token_parser

FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

df = sotu.load()
docnames = []
for row_position, speech in df.iterrows():
    docnames.append(f"{row_position}_{speech['president']}_{speech['year']}")
print(f'{len(df)} speeches, {df["year"].min()} to {df["year"].max()}.')

In [ ]:
# Gap 1 (D2-PM pp10-11): LIWC summary metrics over time. WC (word count) and
# WPS (words per sentence) come straight from liwcalike and do not depend on the
# dictionary, so the open macdvirtue.dic is enough to compute them.
summary = liwcalike(list(df['text']), docnames, 'dictionaries/macdvirtue.dic')
summary['year'] = list(df['year'])
summary = summary.sort_values('year')

metrics = [
    ('WC', 'Word count', 'State of the Union: words per address', 'slide-sotu-wc'),
    ('WPS', 'Words per sentence', 'State of the Union: words per sentence', 'slide-sotu-wps'),
]
for column, ylabel, title, name in metrics:
    plt.figure(figsize=(13, 4))
    plt.scatter(summary['year'], summary[column], s=18)
    plt.xlabel('Year')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Gap 2 (D3-AM p11): nuclear-vocabulary co-occurrence over the WHOLE SOTU corpus.
# Notebook 3a builds one network per decade; this is the all-speeches version.
# save_path lets the helper write the PNG before it shows the figure, and its
# layout is seeded (seed=42), so the image is reproducible.
nuke_token_categories, nuke_categories = load_token_parser('dictionaries/nuke.dic')
nuke_df = liwcalike(list(df['text']), docnames, 'dictionaries/nuke.dic')
counts = nuke_df[nuke_categories].to_numpy()

draw_cooccurrence_network(
    counts,
    nuke_categories,
    top_edges=40,
    node_color='lightyellow',
    title='Nuclear vocabulary co-occurrence (whole State of the Union corpus)',
    save_path=FIGURES / 'slide-sotu-nuke-network.png',
)